# LLM Ontology Mapper Playground

Use this notebook to experiment with the local package at `../` (project root).

**Before opening this notebook**, run once from your terminal:
```bash
uv sync --extra dev --extra openai --extra eval
```
Then open the notebook with the project `.venv` as the kernel
(VS Code: click the kernel picker top-right → select `.venv`).

Suggested run order:
1. Run cell 2 — environment check (confirm Python is from `.venv`)
2. Run cell 3 — install step (only needed if `uv sync` was not run first)
3. Run cell 4 — verify imports
4. Run remaining cells to explore the API


In [1]:
from pathlib import Path
import sys

PACKAGE_ROOT = Path('../')
print(f'Python executable: {sys.executable}')
print(f'Python version: {sys.version.split()[0]}')
print(f'Package root exists: {PACKAGE_ROOT.exists()} -> {PACKAGE_ROOT}')

Python executable: /Users/akandwal/Developer/llm-ontology-mapper/.venv/bin/python
Python version: 3.14.4
Package root exists: True -> ..


In [3]:
# Install the local package in editable mode if needed.
# uv-managed venvs do not include pip, so use uv directly.
#
# Preferred: run this once from your terminal before opening the notebook:
#   uv sync --extra dev --extra openai --extra eval
#
# Or run the cell below to install from inside the notebook:
import subprocess, sys
result = subprocess.run(
    ["uv", "pip", "install", "-e", "../."],
    capture_output=True, text=True
)
print(result.stdout or result.stderr or "Install complete.")


Using Python 3.14.4 environment at: /Users/akandwal/Developer/llm-ontology-mapper/.venv
Resolved 17 packages in 6ms
   Building llm-ontology-mapper @ file:///Users/akandwal/Developer/llm-ontology-mapper
      Built llm-ontology-mapper @ file:///Users/akandwal/Developer/llm-ontology-mapper
Prepared 1 package in 277ms
Uninstalled 1 package in 1ms
Installed 1 package in 2ms
 ~ llm-ontology-mapper==0.1.0 (from file:///Users/akandwal/Developer/llm-ontology-mapper)



In [4]:
import importlib
import inspect
import sys

# Force-refresh local package modules so notebook reruns pick up source edits.
for module_name in list(sys.modules):
    if module_name == 'llm_ontology_mapper' or module_name.startswith('llm_ontology_mapper.'):
        sys.modules.pop(module_name)

import llm_ontology_mapper as lom
from llm_ontology_mapper import OntologyMapper, OntologyRetriever, MappingBatch, MappingResult

print('Package import OK')
print('Package version:', getattr(lom, '__version__', 'unknown'))
print('Package file:', getattr(lom, '__file__', 'unknown'))
print('Exported API:', [n for n in ['OntologyMapper', 'OntologyRetriever', 'MappingBatch', 'MappingResult', 'NERQueryExtractor'] if hasattr(lom, n)])

print('\nOntologyMapper.__init__ signature:')
print(inspect.signature(OntologyMapper.__init__))

print('\nOntologyMapper.map_term signature:')
print(inspect.signature(OntologyMapper.map_term))

print('\nOntologyMapper.map_data_dictionary signature:')
print(inspect.signature(OntologyMapper.map_data_dictionary))

Package import OK
Package version: 0.1.0
Package file: /Users/akandwal/Developer/llm-ontology-mapper/src/llm_ontology_mapper/__init__.py
Exported API: ['OntologyMapper', 'OntologyRetriever', 'MappingBatch', 'MappingResult', 'NERQueryExtractor']

OntologyMapper.__init__ signature:
(self, provider: 'str' = 'openai', model: 'str' = 'gpt-4o', api_key: 'str | None' = None, llm_provider: 'BaseLLMProvider | None' = None, ontologies: 'list[str] | None' = None, ontology_config_path: 'str | None' = None, cache_dir: 'str | None' = '.ontology_cache', use_rag: 'bool' = False, ontology_retriever: 'Any | None' = None, rag_top_k: 'int' = 5, rag_auto_accept_threshold: 'float' = 0.0, use_ontogpt: 'bool' = False, **provider_kwargs: 'Any') -> 'None'

OntologyMapper.map_term signature:
(self, source_term: 'str', source_label: 'str | None' = None, source_type: 'str | None' = None, entity_type: 'str | None' = None) -> 'MappingResult'

OntologyMapper.map_data_dictionary signature:
(self, records: 'list[dict[s

In [5]:
# Helper for resilient display while package internals are still evolving
def show_mapping_result(result):
    fields = [
        'source_term', 'source_label', 'source_type',
        'target_code', 'target_term', 'ontology',
        'confidence', 'logic_type', 'notes',
    ]
    print('--- Mapping Result ---')
    for f in fields:
        if hasattr(result, f):
            print(f'{f}:', getattr(result, f))
    # Some in-progress branches may expose a curie property/field
    if hasattr(result, 'curie'):
        print('curie:', getattr(result, 'curie'))


def create_mapper(provider='openai', model='gpt-4o-mini', use_rag=False, retriever=None, **provider_kwargs):
    """Create OntologyMapper with provider-specific kwargs support."""
    kwargs = {'provider': provider, 'model': model, 'use_rag': use_rag, **provider_kwargs}
    if retriever is not None:
        # Current constructor parameter name in mapper.py
        kwargs['ontology_retriever'] = retriever
    return OntologyMapper(**kwargs)

print('Helpers loaded.')

Helpers loaded.


In [6]:
# Optional: inspect available submodules
import pkgutil
import llm_ontology_mapper as lom

if hasattr(lom, '__path__'):
    submods = [m.name for m in pkgutil.iter_modules(lom.__path__)]
    print('Submodules:')
    for mod in submods:
        print('-', mod)
else:
    print('Package does not expose __path__.')

Submodules:
- evaluator
- mapper
- models
- ner_extractor
- providers
- retriever
- validator


## Concrete usage examples (from package source)

These examples are tailored to the current public API in your package:
- `OntologyMapper.map_term(source_term, source_label, source_type, entity_type)`
- `OntologyMapper.map_data_dictionary(records, ...)`
- RAG mode via `use_rag=True` + `ontology_retriever=OntologyRetriever(...)`

Before running examples:
1. Make sure package install/import cells succeeded.
2. Configure provider credentials in the next code cell.
3. If your branch is mid-refactor, keep using the helper cell output to confirm signatures.

### Provider credential checklist

- OpenAI
  - Set `OPENAI_API_KEY`
  - Optional custom endpoint: `OPENAI_BASE_URL`
- GitHub Models
  - Set `GITHUB_TOKEN`
  - Optional override endpoint: `GITHUB_BASE_URL` (default is `https://models.inference.ai.azure.com`)
- Ollama local (recommended default)
  - Leave `OLLAMA_BASE_URL` unset to use local server default
  - Start Ollama locally (usually `http://localhost:11434`)
  - Pull model first (for example `ollama pull llama3`)
- Ollama cloud/remote
  - Set `OLLAMA_BASE_URL` (for example `https://api.ollama.com` or your VM URL)
  - If endpoint requires auth, set `OLLAMA_API_KEY` (sent as Bearer token)

### Graceful `base_url` strategy

The next code cell builds provider kwargs from environment variables so you can switch providers without editing mapping examples.

In [ ]:
# Provider setup + credential validation for OpenAI, GitHub Models, and Ollama
import os

# ─── In-notebook credential overrides ────────────────────────────────────────
# Jupyter kernels only see env vars that existed when the kernel STARTED.
# If you set env vars in the terminal AFTER launching Jupyter, set them here
# instead of restarting the kernel. Leave a value as '' to keep the current env.
_INLINE_CREDENTIALS = {
    'OLLAMA_API_KEY':  '',   # e.g. 'your-ollama-token'
    'OLLAMA_BASE_URL': 'https://ollama.com',   # e.g. 'https://ollama.com' or 'http://your-vm:11434'
    'OLLAMA_MODEL':    '',   # e.g. 'gpt-oss:120b'
    'OPENAI_API_KEY':  '',
    'GITHUB_TOKEN':    '',
}
for _k, _v in _INLINE_CREDENTIALS.items():
    if _v:
        os.environ[_k] = _v
        print(f'Set {_k} from in-notebook override.')
# ──────────────────────────────────────────────────────────────────────────────

def _has_env(name: str) -> bool:
    return bool(os.getenv(name, '').strip())

# Choose the provider here in the notebook: 'openai', 'github', or 'ollama'.
SELECTED_PROVIDER = 'openai'
PROVIDER_SOURCE = 'notebook cell'

# Recommended default models (adjust as needed)
DEFAULT_MODELS = {
    'openai': 'gpt-4o-mini',
    'github': 'gpt-4o-mini',
    'ollama': 'gpt-oss:120b',
}

# Centralized endpoint configuration (graceful base_url fallback)
DEFAULT_BASE_URLS = {
    'openai': os.getenv('OPENAI_BASE_URL', '').strip(),  # blank -> official OpenAI endpoint
    'github': os.getenv('GITHUB_BASE_URL', 'https://models.inference.ai.azure.com').strip(),
    # blank -> OllamaProvider uses local default (localhost:11434)
    # set OLLAMA_BASE_URL=https://ollama.com for cloud
    'ollama': os.getenv('OLLAMA_BASE_URL', '').strip(),
}

def _normalize_ollama_base_url(url: str) -> str:
    """Ollama client appends /api/chat, so base_url must not include /api."""
    if not url:
        return url
    normalized = url.strip().rstrip('/')
    if normalized.endswith('/api'):
        normalized = normalized[:-4].rstrip('/')
        print(f'Normalized OLLAMA_BASE_URL: removed trailing /api -> {normalized!r}')
    return normalized

def provider_kwargs_from_env(provider: str) -> dict:
    provider = provider.lower()
    base_url = DEFAULT_BASE_URLS.get(provider, '')
    kwargs = {}

    if provider == 'ollama':
        base_url = _normalize_ollama_base_url(base_url)

    # Include base_url only when configured (or defaulted, e.g. GitHub)
    if base_url:
        kwargs['base_url'] = base_url

    # Provider-specific auth handling
    if provider == 'ollama':
        # Bearer token for Ollama cloud or auth-protected VM
        token = os.getenv('OLLAMA_API_KEY', '').strip()
        if token:
            kwargs['api_key'] = token

    return kwargs

PROVIDER_KWARGS = {
    'openai': provider_kwargs_from_env('openai'),
    'github': provider_kwargs_from_env('github'),
    'ollama': provider_kwargs_from_env('ollama'),
}

# Credential checks
required_env = {
    'openai': ['OPENAI_API_KEY'],
    'github': ['GITHUB_TOKEN'],
    # Ollama cloud requires OLLAMA_API_KEY; local/VM typically does not
    'ollama': ['OLLAMA_API_KEY'] if PROVIDER_KWARGS.get('ollama', {}).get('base_url', '').lower().endswith('ollama.com') else [],
}

assert SELECTED_PROVIDER in DEFAULT_MODELS, f'Unsupported provider: {SELECTED_PROVIDER}'
missing = [k for k in required_env.get(SELECTED_PROVIDER, []) if not _has_env(k)]

# Warn if Ollama cloud URL is set but no token provided
ollama_url = PROVIDER_KWARGS.get('ollama', {}).get('base_url', '')
using_ollama_cloud = bool(ollama_url) and any(h in ollama_url.lower() for h in ('ollama.com', 'api.ollama'))
if SELECTED_PROVIDER == 'ollama' and using_ollama_cloud and not _has_env('OLLAMA_API_KEY'):
    missing.append('OLLAMA_API_KEY')

print('Selected provider:', SELECTED_PROVIDER)
print('Provider selected from:', PROVIDER_SOURCE)
print('Model:', DEFAULT_MODELS[SELECTED_PROVIDER])
print('Provider kwargs:', PROVIDER_KWARGS.get(SELECTED_PROVIDER, {}))
print('OPENAI_API_KEY visible to kernel:', _has_env('OPENAI_API_KEY'))
print('GITHUB_TOKEN visible to kernel:', _has_env('GITHUB_TOKEN'))
if SELECTED_PROVIDER == 'ollama':
    print('OLLAMA_API_KEY set:', _has_env('OLLAMA_API_KEY'))
if missing:
    print('Missing environment variables:', ', '.join(missing))
    print('Set them via _INLINE_CREDENTIALS above, or restart Jupyter from a shell where the env is already exported.')
else:
    print('Credential check passed.')

Set OLLAMA_BASE_URL from in-notebook override.
Set OPENAI_API_KEY from in-notebook override.
Selected provider: openai
Provider selected from: notebook cell
Model: gpt-4o-mini
Provider kwargs: {}
OPENAI_API_KEY visible to kernel: True
GITHUB_TOKEN visible to kernel: False
Credential check passed.


In [8]:
# Smoke test: instantiate mapper and run one lightweight term mapping
provider = SELECTED_PROVIDER
model = DEFAULT_MODELS[provider]
provider_kwargs = PROVIDER_KWARGS.get(provider, {})

effective_base_url = provider_kwargs.get('base_url', '(provider default/local ollama)')
print(f'Provider={provider} | Model={model} | base_url={effective_base_url}')

mapper = create_mapper(
    provider=provider,
    model=model,
    use_rag=False,
    **provider_kwargs,
 )

print(f'Created mapper with provider={provider}, model={model}')

try:
    smoke_result = mapper.map_term(
        source_term='bp_sys',
        source_label='Systolic Blood Pressure',
        source_type='integer',
        entity_type='phenotype',
    )
    show_mapping_result(smoke_result)
except RuntimeError as exc:
    msg = str(exc)
    raise

Provider=openai | Model=gpt-4o-mini | base_url=(provider default/local ollama)
Created mapper with provider=openai, model=gpt-4o-mini
--- Mapping Result ---
source_term: bp_sys
source_label: Systolic Blood Pressure
source_type: integer
target_code: HP:0002027
target_term: Abnormal blood pressure
ontology: HPO
confidence: 0.9
logic_type: LogicType.LLM
notes: Mapped


In [9]:
# Example 1: Single-term mapping (non-RAG)

mapper = create_mapper(
    provider=SELECTED_PROVIDER,
    model=DEFAULT_MODELS[SELECTED_PROVIDER],
    use_rag=False,
    **PROVIDER_KWARGS.get(SELECTED_PROVIDER, {}),
)

single_result = mapper.map_term(
    source_term='systolic_bp',
    source_label='Systolic Blood Pressure',
    source_type='integer',
    entity_type='phenotype',
)

show_mapping_result(single_result)

--- Mapping Result ---
source_term: systolic_bp
source_label: Systolic Blood Pressure
source_type: integer
target_code: HP:0002020
target_term: Abnormality of blood pressure
ontology: HPO
confidence: 0.9
logic_type: LogicType.LLM
notes: Mapped


In [10]:
# Example 2: Batch mapping with map_data_dictionary

records = [
    {'field_name': 'age_at_diagnosis', 'field_label': 'Age at Diagnosis', 'field_type': 'integer'},
    {'field_name': 'smoking_status', 'field_label': 'Smoking Status', 'field_type': 'categorical'},
    {'field_name': 'bmi', 'field_label': 'Body Mass Index', 'field_type': 'float'},
]

batch_result = mapper.map_data_dictionary(
    records=records,
    source_term_field='field_name',
    source_label_field='field_label',
    source_type_field='field_type',
    entity_type='phenotype',
    study_id='demo-study-001',
)

print('Batch type:', type(batch_result).__name__)
print('Total mapped:', len(getattr(batch_result, 'results', [])))
for i, r in enumerate(getattr(batch_result, 'results', []), start=1):
    print(f'\nResult {i}')
    show_mapping_result(r)

Batch type: MappingBatch
Total mapped: 3

Result 1
--- Mapping Result ---
source_term: age_at_diagnosis
source_label: Age at Diagnosis
source_type: integer
target_code: HP:0000005
target_term: Age at onset
ontology: HPO
confidence: 0.9
logic_type: LogicType.LLM
notes: Mapped

Result 2
--- Mapping Result ---
source_term: smoking_status
source_label: Smoking Status
source_type: categorical
target_code: HP:0000007
target_term: Smoking
ontology: HPO
confidence: 0.9
logic_type: LogicType.LLM
notes: Mapped

Result 3
--- Mapping Result ---
source_term: bmi
source_label: Body Mass Index
source_type: float
target_code: HP:0001945
target_term: Body mass index
ontology: HPO
confidence: 0.9
logic_type: LogicType.LLM
notes: Mapped


In [11]:
# Example 3: RAG-enabled single-term mapping

retriever = OntologyRetriever(
    ontologies=['HPO', 'MONDO', 'NCIT'],
    top_k=5,
    cache_enabled=True,
    api_timeout=10,
)

mapper_rag = create_mapper(
    provider=SELECTED_PROVIDER,
    model=DEFAULT_MODELS[SELECTED_PROVIDER],
    use_rag=True,
    retriever=retriever,
    **PROVIDER_KWARGS.get(SELECTED_PROVIDER, {}),
)

rag_result = mapper_rag.map_term(
    source_term='dx_cancer',
    source_label='Cancer Diagnosis',
    source_type='text',
    entity_type='diagnosis',
)

show_mapping_result(rag_result)

--- Mapping Result ---
source_term: dx_cancer
source_label: Cancer Diagnosis
source_type: text
target_code: NCIT:C16213
target_term: Cancer Diagnosis
ontology: NCIT
confidence: 1.0
logic_type: LogicType.RAG
notes: Mapped


In [ ]:
# Example 4: Optional NER-assisted retrieval (if scispaCy models are installed)

from llm_ontology_mapper import NERQueryExtractor

ner = NERQueryExtractor()
print('NER available:', ner.is_available())

if ner.is_available():
    retriever_ner = OntologyRetriever(
        ontologies=['HPO', 'MONDO', 'NCIT'],
        top_k=5,
        ner_extractor=ner,
    )
    mapper_rag_ner = create_mapper(
        provider=SELECTED_PROVIDER,
        model=DEFAULT_MODELS[SELECTED_PROVIDER],
        use_rag=True,
        retriever=retriever_ner,
        **PROVIDER_KWARGS.get(SELECTED_PROVIDER, {}),
    )
    ner_result = mapper_rag_ner.map_term(
        source_term='htn_diag_age',
        source_label='Age at hypertension diagnosis',
        source_type='integer',
        entity_type='diagnosis',
    )
    show_mapping_result(ner_result)
else:
    print('Install scispacy + model weights to enable this example.')